# Train โมเดลด้วย GPU บน Google Colab

Notebook นี้เตรียมไว้สำหรับผู้เริ่มต้น: เลือก GPU แล้วกด **Run all** เพื่อ Clone โปรเจกต์ ตรวจ Dataset ทดสอบระบบ และ Train Model ที่เลือกไว้เพียงหนึ่ง Model

## 1. ตั้งค่าการ Train

เปลี่ยนค่าได้ตามต้องการ โดย `MODEL` ต้องเป็น `mobilenetv2`, `efficientnetb0` หรือ `resnet50` เท่านั้น

In [ ]:
MODEL = "mobilenetv2"
EPOCHS = 10
BATCH_SIZE = 16

ALLOWED_MODELS = {"mobilenetv2", "efficientnetb0", "resnet50"}
if MODEL not in ALLOWED_MODELS:
    raise ValueError(f"MODEL ต้องเป็นหนึ่งใน {sorted(ALLOWED_MODELS)}")
if EPOCHS < 1 or BATCH_SIZE < 1:
    raise ValueError("EPOCHS และ BATCH_SIZE ต้องมากกว่า 0")
print(f"Model: {MODEL} | Epochs: {EPOCHS} | Batch size: {BATCH_SIZE}")

## 2. ตรวจสอบ GPU

ถ้าไม่พบ GPU ให้เลือก `Runtime > Change runtime type > GPU` แล้วเริ่ม Run all ใหม่ การ Train จริงจะไม่เริ่มถ้ายังไม่พบ GPU

In [ ]:
import tensorflow as tf

GPUS = tf.config.list_physical_devices("GPU")
print(GPUS)
GPU_READY = bool(GPUS)
if GPU_READY:
    print("✅ GPU พร้อมใช้งาน")
else:
    print("❌ ไม่พบ GPU")
    print("กรุณาเลือก Runtime > Change runtime type > GPU แล้ว Run all ใหม่")

## 3. Clone หรืออัปเดต Repository

Cell นี้รันซ้ำได้ ถ้ามี Repository อยู่แล้วจะอัปเดตด้วย `git pull --ff-only`

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Bluedabade/ML-Assignment.git"
PROJECT_DIR = Path("/content/ML-Assignment")
if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print(f"โฟลเดอร์โปรเจกต์: {Path.cwd()}")

## 4. ตรวจและติดตั้ง Library ที่ขาด

Colab มี TensorFlow พร้อม GPU อยู่แล้ว Cell นี้จึง **ไม่ติดตั้ง TensorFlow, Keras, CUDA หรือ cuDNN ซ้ำ** และจะติดตั้งเฉพาะ Library อื่นที่ยังขาด

In [ ]:
import importlib.util
import sys

REQUIRED = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "seaborn": "seaborn",
    "PIL": "Pillow", "tqdm": "tqdm",
}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("ติดตั้ง Library ที่ขาดแล้ว:", ", ".join(missing))
else:
    print("Library ที่จำเป็นพร้อมใช้งานแล้ว")
print("TensorFlow:", tf.__version__)

## 5. ตรวจสอบ Dataset และ Class

ใช้ Dataset ที่อยู่ใน `data/processed` โดยตรง ไม่ดาวน์โหลดจาก Kaggle หรือ Roboflow

In [ ]:
EXPECTED_CLASSES = ["normal", "wrinkle", "acne", "dark_spot", "large_pores"]
EXPECTED_TOTALS = {"train": 1712, "validation": 366, "test": 372}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
DATA_DIR = PROJECT_DIR / "data" / "processed"

for split, expected_total in EXPECTED_TOTALS.items():
    split_dir = DATA_DIR / split
    classes = [path.name for path in split_dir.iterdir() if path.is_dir()]
    if set(classes) != set(EXPECTED_CLASSES):
        raise RuntimeError(f"{split}: Class ไม่ตรงตามที่กำหนด: {classes}")
    total = sum(1 for path in split_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    print(f"{split}: {total} images | classes: {EXPECTED_CLASSES}")
    if total != expected_total:
        raise RuntimeError(f"{split}: ควรมี {expected_total} images แต่พบ {total}")

from config import CLASS_NAMES
if CLASS_NAMES != EXPECTED_CLASSES:
    raise RuntimeError(f"Class mapping ของโปรเจกต์ไม่ตรง: {CLASS_NAMES}")
print("✅ Dataset และ Class mapping ถูกต้องครบ 5 classes")

## 6. ทดสอบระบบก่อน Train จริง

Smoke test ใช้ข้อมูลเพียงไม่กี่ batch เพื่อตรวจว่า Dataset, Model และ Environment ทำงานได้ ผลนี้ **ไม่ใช่ผลการ Train ขั้นสุดท้าย**

In [ ]:
if GPU_READY:
    subprocess.run([sys.executable, "train.py", "--model", "mobilenetv2", "--batch-size", str(BATCH_SIZE), "--smoke-test"], check=True)
else:
    print("⏭️ ข้าม Smoke test เพราะไม่พบ GPU")

## 7. Train Model ที่เลือก

เมื่อกด Run all จะ Train เฉพาะ Model ที่ตั้งไว้ใน `MODEL` ไม่ได้ Train ทั้งสาม Model พร้อมกัน TensorFlow จะใช้ GPU ที่ Colab ตรวจพบให้อัตโนมัติ

In [ ]:
if GPU_READY:
    command = [
        sys.executable, "train.py", "--model", MODEL,
        "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
    ]
    print("กำลังรัน:", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("⏭️ ไม่เริ่ม Train เพราะไม่พบ GPU กรุณาเปลี่ยน Runtime เป็น GPU แล้ว Run all ใหม่")

## 8. แสดงผลลัพธ์ที่สร้างเสร็จแล้ว

จะแสดงกราฟ, Metrics และ Confusion Matrix เฉพาะไฟล์ที่มีอยู่จริงเท่านั้น ไม่มีการสร้างผลลัพธ์จำลอง

In [ ]:
import json
from IPython.display import Image, display

plot_dir = PROJECT_DIR / "reports" / "plots"
result_dir = PROJECT_DIR / "reports" / "results" / MODEL
for title, path in [
    ("Training Accuracy vs Validation Accuracy", plot_dir / f"{MODEL}_accuracy.png"),
    ("Training Loss vs Validation Loss", plot_dir / f"{MODEL}_loss.png"),
]:
    if path.exists():
        print("\n" + title)
        display(Image(filename=str(path)))
    else:
        print(f"ยังไม่พบ {path.name}")

metrics_path = result_dir / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("\nผลประเมิน Test set")
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['macro_precision']:.4f}")
    print(f"Recall:    {metrics['macro_recall']:.4f}")
    print(f"F1-score:  {metrics['macro_f1']:.4f}")
else:
    print("ยังไม่มี Metrics จากการ Train ที่เสร็จสมบูรณ์")

confusion_path = result_dir / "confusion_matrix.png"
if confusion_path.exists():
    print("\nConfusion Matrix")
    display(Image(filename=str(confusion_path)))
else:
    print("ยังไม่มี Confusion Matrix")

## 9. ดาวน์โหลดผลลัพธ์ (ตัวเลือก)

ไฟล์ใน Colab จะหายเมื่อ Runtime ถูกรีเซ็ต หากต้องการเก็บ Model, Metrics และกราฟ ให้เปลี่ยน `DOWNLOAD_RESULTS = True` แล้วรัน Cell นี้เอง Cell นี้จะไม่ commit ไฟล์กลับ GitHub

In [ ]:
DOWNLOAD_RESULTS = False

if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files

    export_dir = PROJECT_DIR / "colab_export" / MODEL
    export_dir.mkdir(parents=True, exist_ok=True)
    candidates = [
        PROJECT_DIR / "artifacts" / MODEL / "best_model.keras",
        result_dir / "metrics.json", result_dir / "classification_report.txt",
        result_dir / "confusion_matrix.png", result_dir / "history.json",
        plot_dir / f"{MODEL}_accuracy.png", plot_dir / f"{MODEL}_loss.png",
    ]
    copied = 0
    for path in candidates:
        if path.exists():
            shutil.copy2(path, export_dir / path.name)
            copied += 1
    if copied:
        archive = shutil.make_archive(str(PROJECT_DIR / f"{MODEL}_colab_results"), "zip", export_dir)
        files.download(archive)
    else:
        print("ยังไม่มีผลลัพธ์สำหรับดาวน์โหลด")
else:
    print("ไม่ได้ดาวน์โหลดอัตโนมัติ เปลี่ยน DOWNLOAD_RESULTS เป็น True เมื่อต้องการเก็บไฟล์")